# Talha - Temel Yapay Zeka Eğitimi
# Talha - Basic AI Training

Bu notebook, Talha projesinde temel bir sinir ağı modelinin nasıl eğitileceğini gösterir.

This notebook demonstrates how to train a basic neural network model in the Talha project.

## 1. Kütüphaneleri İçe Aktar / Import Libraries

In [ ]:
import sys
sys.path.append('..')

import torch
import matplotlib.pyplot as plt
import numpy as np

from src.talha.model import SimpleNeuralNetwork
from src.talha.data_loader import load_data, create_synthetic_data
from src.talha.trainer import Trainer
from src.talha.config import MODEL_CONFIG, TRAINING_CONFIG

print("Kütüphaneler başarıyla yüklendi! / Libraries loaded successfully!")
print(f"PyTorch versiyonu / PyTorch version: {torch.__version__}")
print(f"CUDA kullanılabilir mi? / CUDA available?: {torch.cuda.is_available()}")

## 2. Veri Yükleme / Data Loading

MNIST veri setini yükleyelim.

In [ ]:
# Veri yükleyicileri oluştur / Create data loaders
print("Veri yükleniyor... / Loading data...")
train_loader, val_loader, test_loader = load_data(batch_size=32)

print(f"Eğitim örnekleri / Training samples: {len(train_loader.dataset)}")
print(f"Doğrulama örnekleri / Validation samples: {len(val_loader.dataset)}")
print(f"Test örnekleri / Test samples: {len(test_loader.dataset)}")

### Veri Görselleştirme / Data Visualization

In [ ]:
# İlk batch'ten bazı örnekleri görselleştir
# Visualize some examples from the first batch

dataiter = iter(train_loader)
images, labels = next(dataiter)

fig, axes = plt.subplots(2, 5, figsize=(12, 6))
axes = axes.ravel()

for i in range(10):
    axes[i].imshow(images[i].squeeze(), cmap='gray')
    axes[i].set_title(f'Etiket / Label: {labels[i].item()}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 3. Model Oluşturma / Model Creation

In [ ]:
# Model oluştur / Create model
model = SimpleNeuralNetwork(
    input_size=MODEL_CONFIG["input_size"],
    hidden_sizes=MODEL_CONFIG["hidden_sizes"],
    output_size=MODEL_CONFIG["output_size"],
    dropout_rate=MODEL_CONFIG["dropout_rate"]
)

print("Model Mimarisi / Model Architecture:")
print(model)
print(f"\nToplam parametreler / Total parameters: {sum(p.numel() for p in model.parameters()):,}")

## 4. Model Eğitimi / Model Training

In [ ]:
# Trainer oluştur / Create trainer
config = TRAINING_CONFIG.copy()
config["num_epochs"] = 5  # Notebook için daha az epoch

trainer = Trainer(model, config=config)
print(f"Eğitim cihazı / Training device: {trainer.device}")

In [ ]:
# Eğitimi başlat / Start training
trainer.train(train_loader, val_loader, num_epochs=5)

## 5. Eğitim Sonuçlarını Görselleştirme / Visualize Training Results

In [ ]:
# Kayıp grafiği / Loss plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Kayıp / Loss
ax1.plot(trainer.history['train_loss'], label='Eğitim Kaybı / Train Loss')
ax1.plot(trainer.history['val_loss'], label='Doğrulama Kaybı / Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Kayıp / Loss')
ax1.set_title('Model Kaybı / Model Loss')
ax1.legend()
ax1.grid(True)

# Doğruluk / Accuracy
ax2.plot(trainer.history['train_accuracy'], label='Eğitim Doğruluğu / Train Acc')
ax2.plot(trainer.history['val_accuracy'], label='Doğrulama Doğruluğu / Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Doğruluk (%) / Accuracy (%)')
ax2.set_title('Model Doğruluğu / Model Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 6. Test Seti Değerlendirmesi / Test Set Evaluation

In [ ]:
# Test seti üzerinde değerlendirme / Evaluate on test set
test_loss, test_acc = trainer.validate(test_loader)

print(f"Test Kaybı / Test Loss: {test_loss:.4f}")
print(f"Test Doğruluğu / Test Accuracy: {test_acc:.2f}%")

## 7. Tahmin Örnekleri / Prediction Examples

In [ ]:
# Test setinden bazı tahminleri görselleştir
# Visualize some predictions from test set

model.eval()
test_images, test_labels = next(iter(test_loader))
test_images = test_images.to(trainer.device)

with torch.no_grad():
    outputs = model(test_images)
    _, predicted = torch.max(outputs, 1)

# Görselleştirme / Visualization
fig, axes = plt.subplots(2, 5, figsize=(15, 7))
axes = axes.ravel()

for i in range(10):
    img = test_images[i].cpu().squeeze()
    true_label = test_labels[i].item()
    pred_label = predicted[i].item()
    
    axes[i].imshow(img, cmap='gray')
    color = 'green' if true_label == pred_label else 'red'
    axes[i].set_title(f'Gerçek: {true_label}\nTahmin: {pred_label}', color=color)
    axes[i].axis('off')

plt.tight_layout()
plt.show()

## 8. Model Kaydetme / Save Model

In [ ]:
# Modeli kaydet / Save model
trainer.save_model("notebook_model.pth")
trainer.save_history("notebook_history.json")

print("Model ve eğitim geçmişi kaydedildi! / Model and training history saved!")

## Sonuç / Conclusion

Bu notebook'ta:
1. MNIST veri setini yükledik
2. Basit bir sinir ağı modeli oluşturduk
3. Modeli eğittik
4. Sonuçları görselleştirdik
5. Test seti üzerinde değerlendirme yaptık

In this notebook we:
1. Loaded the MNIST dataset
2. Created a simple neural network model
3. Trained the model
4. Visualized the results
5. Evaluated on the test set